# APK 6725 - AI for Sport & Movement Sciences
## Assignment 6: Linear Regression and Model Evaluation

This assignment practices the ideas from lectures 03.01 and 03.02. You will build linear regression models that predict walking speed from gait measures. You will then check each model on data it did not see during training. Write your code in the cells provided and test it.

Total: 5 points.

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
os.chdir('/content')                      # start from a valid directory
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/AIforMovementScience'
if os.path.isdir(REPO + '/.git'):
    # already downloaded before: just update it
    !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
else:
    # first time, or a broken/partial copy: download a fresh copy
    !rm -rf "$REPO"
    !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"

WORKDIR = REPO + '/Assignments'          # this notebook's folder in the course files
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)           # so local .py files (e.g. FunctionExamples.py) import correctly
DATA = REPO + '/lectures/data'        # the folder that holds this assignment's data files
print('Setup complete. You can now run the rest of the notebook.')

## The data

The file **`gait_measures_PD_controls_ALL.csv`** comes from the Gait in Parkinson's Disease database (PhysioNet, <https://physionet.org/content/gaitpdb/>). You used it in Assignment 4. It is in the course files, in `lectures/data`. The setup cell stored that path in the variable `DATA`.

Each row is one walking recording. The file has 18 measures per recording:

| Column | Contents |
| --- | --- |
| `ID` | the subject who produced the recording |
| `class` | `0` for a control subject, `1` for a subject with Parkinson's disease |
| `avg_speed` | walking speed, in m/s |
| `avg_step_length` | step length, in m |
| `cadence` | steps per minute |
| `avg_stride_time_left`, `avg_stride_time_right` | time of one stride, in s |
| `avg_stance_time_left`, `avg_stance_time_right` | time the foot is on the floor, in s |
| `avg_swing_time_left`, `avg_swing_time_right` | time the foot is in the air, in s |
| `avg_step_time` | time of one step, in s |
| `cv_...` (7 columns) | variability of each timing measure (coefficient of variation) |

**Goal:** predict `avg_speed` from the other gait measures. Speed is the output. The `class` column is not an input. The model must use gait measures only.

## How to submit

Before you submit, please make sure your notebook is organized and reproducible. Part of your grade is for this.

- Answer each question in its own section, using the headings already provided.
- Put your code in the code cells; when a question asks for a written answer, use the markdown answer cell.
- Delete any scratch or experimental cells that are not part of your answer.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_6.ipynb`, and upload it to Canvas.

## Question 1 (0.5 points)

Load the data and prepare it for modeling. Answer parts (a) and (b), each in its own cell.

### Part (a) - Load and clean the table

Read `gait_measures_PD_controls_ALL.csv` into a DataFrame called `data`. The file is at `f"{DATA}/gait_measures_PD_controls_ALL.csv"`.

- Print the shape of the table.
- Print the number of missing values in each column that has any.
- Remove every row with a missing value.
- Print the new shape and the number of different subjects (the number of unique values in `ID`).

In the answer cell: how many rows did you remove? How can 305 rows come from fewer subjects?

In [ ]:
import pandas as pd

data = pd.read_csv(f"{DATA}/gait_measures_PD_controls_ALL.csv")
print('shape:', data.shape)

missing = data.isna().sum()
print('\nmissing values per column (only columns that have any):')
print(missing[missing > 0])

n_before = len(data)
data = data.dropna()
print('\nnew shape:', data.shape)
print('rows removed:', n_before - len(data))
print('number of different subjects:', data['ID'].nunique())

**Your answer:** I removed **1 row**. Recording `GaPt03` (row 38) was missing both `avg_speed` and `avg_step_length`. The table went from 306 to 305 rows.

The 305 rows come from only **164 different subjects** because each row is one walking *recording*, not one person, and many subjects were recorded more than once. 110 subjects have one recording, while others have 2, 3, 5, 6 or even 7 (`data['ID'].value_counts()` shows this). So several rows can belong to the same person.

### Part (b) - Inputs, output and split

Make a list called `features` with every column except `ID`, `class` and `avg_speed`. Print how many features it holds.

Split the **whole DataFrame** into a train set (75%) and a test set (25%). Use `train_test_split` with `random_state=42`. Splitting the whole DataFrame keeps the `ID` of each row, and you need it in Question 7. Then create:

- `X_train` and `X_test`: the `features` columns of each set
- `y_train` and `y_test`: the `avg_speed` column of each set

Print the number of rows in each set.

In [ ]:
from sklearn.model_selection import train_test_split

features = [col for col in data.columns if col not in ['ID', 'class', 'avg_speed']]
print('number of features:', len(features))
print(features)

# split the whole DataFrame so that each row keeps its ID
train, test = train_test_split(data, test_size=0.25, random_state=42)

X_train = train[features]
X_test  = test[features]
y_train = train['avg_speed']
y_test  = test['avg_speed']

print('\nrows in the train set:', len(X_train))
print('rows in the test set: ', len(X_test))

## Question 2 (1 point)

Start with the simplest model: one input. Predict `avg_speed` from `avg_step_length`. Answer parts (a) and (b), each in its own cell.

### Part (a) - Fit the model

Train a `LinearRegression` on the train set, with `avg_step_length` as the only input. Print the slope and the intercept.

**Hint:** scikit-learn needs a 2-D input table. Use double brackets, `X_train[['avg_step_length']]`, to keep a table with one column.

In the answer cell:
- Say what the slope means, with its units. For example, how much does the predicted speed change if the step length grows by 0.1 m?
- Use the model to predict the speed of a subject with a step length of 0.70 m.

In [ ]:
from sklearn.linear_model import LinearRegression

model_q2 = LinearRegression()
model_q2.fit(X_train[['avg_step_length']], y_train)

slope = model_q2.coef_[0]
intercept = model_q2.intercept_
print(f'slope:     {slope:.4f} (m/s per m of step length)')
print(f'intercept: {intercept:.4f} m/s')

print(f'\nchange in predicted speed for +0.1 m of step length: {slope * 0.1:.3f} m/s')

new_subject = pd.DataFrame({'avg_step_length': [0.70]})
print(f'predicted speed for a step length of 0.70 m: {model_q2.predict(new_subject)[0]:.3f} m/s')

**Your answer:** The model is *speed = 0.337 + 1.246 × step length*.

- **Slope:** 1.246 (m/s) per m. For every extra metre of step length, the predicted speed goes up by 1.246 m/s. In more realistic terms, if the step length grows by **0.1 m**, the predicted speed goes up by about **0.125 m/s**.
- **Prediction:** a subject with a step length of 0.70 m has a predicted speed of 0.337 + 1.246 × 0.70 ≈ **1.21 m/s**.

### Part (b) - Evaluate the model

Compute the $R^2$ score of the model on the train set and on the test set. Then make a scatter plot of the **test** data (`avg_step_length` on the x-axis, `avg_speed` on the y-axis) and draw the model line on top of it. Label the axes with their units.

In the answer cell: is this a good model? Compare the two scores. Does the plot agree with them?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

r2_train_q2 = model_q2.score(X_train[['avg_step_length']], y_train)
r2_test_q2  = model_q2.score(X_test[['avg_step_length']], y_test)
print(f'R2 train: {r2_train_q2:.3f}')
print(f'R2 test:  {r2_test_q2:.3f}')

# model line over the range of the test data
x_line = pd.DataFrame({'avg_step_length': np.linspace(X_test['avg_step_length'].min(),
                                                      X_test['avg_step_length'].max(), 100)})
y_line = model_q2.predict(x_line)

plt.figure(figsize=(6, 5))
plt.scatter(X_test['avg_step_length'], y_test, alpha=0.7, label='Test data')
plt.plot(x_line['avg_step_length'], y_line, color='red', lw=2, label='Model')
plt.xlabel('Step length (m)')
plt.ylabel('Walking speed (m/s)')
plt.title(f'Speed vs step length (test set, $R^2$ = {r2_test_q2:.2f})')
plt.legend()
plt.grid()
plt.show()

**Your answer:** It is a reasonable but limited model. The $R^2$ is **0.55 on the train set** and **0.71 on the test set**, so step length alone explains roughly half to two-thirds of the variation in speed.

Usually the test score is lower than the train score. Here it is higher, and that does not mean the model is better on new data. The train set happens to contain a few unusual recordings that the line fits badly, which pulls the train score down. One example is `GaCo13`, with a step length of 1.57 m at only 46 steps/min. The test set has fewer of those recordings.

The plot agrees with the scores. The points follow the rising line, so the trend is clear, but there is a wide scatter around it: subjects with the same step length walk at quite different speeds. That is expected, because speed also depends on cadence (speed = cadence × step length / 60), which this model ignores.

## Question 3 (0.5 points)

Now give the model every input. Train a `LinearRegression` with all the columns in `features`. Compute the $R^2$ score and the adjusted $R^2$ score on the train set and on the test set.

The adjusted $R^2$ penalizes a model for having many inputs. Run the cell below to create the function that computes it.

In [ ]:
def calculate_adjusted_r2(r2_score, n_observations, n_predictors):
    """Adjusted R2. n_observations = rows of X, n_predictors = columns of X."""
    return 1 - (1 - r2_score) * ((n_observations - 1) / (n_observations - n_predictors - 1))

In [ ]:
model_q3 = LinearRegression()
model_q3.fit(X_train, y_train)

r2_train_q3 = model_q3.score(X_train, y_train)
r2_test_q3  = model_q3.score(X_test, y_test)

# n_observations = rows of X, n_predictors = columns of X
adj_train_q3 = calculate_adjusted_r2(r2_train_q3, X_train.shape[0], X_train.shape[1])
adj_test_q3  = calculate_adjusted_r2(r2_test_q3,  X_test.shape[0],  X_test.shape[1])

print(f'train: R2 = {r2_train_q3:.3f}, adjusted R2 = {adj_train_q3:.3f}')
print(f'test:  R2 = {r2_test_q3:.3f}, adjusted R2 = {adj_test_q3:.3f}')

In the answer cell:
- How does this model compare with the one from Question 2?
- Look at the train scores and the test scores. What does the gap between them tell you?

**Your answer:**

- **Comparison with Question 2:** this model is much better. On the test set, $R^2$ rises from 0.71 (step length only) to **0.93** (adjusted 0.92). With all 16 inputs, the model also knows the timing measures (cadence, step and stride times), and speed depends on how often you step as well as how long each step is.
- **Train vs test:** the train scores ($R^2$ = 0.98, adjusted 0.98) are higher than the test scores ($R^2$ = 0.93, adjusted 0.92). This gap means the model fits the data it was trained on a bit better than new data, which is a sign of **mild overfitting**. With 16 inputs, some of them nearly duplicates of each other, the model can adjust to details of the training rows that do not carry over. The adjusted $R^2$ drops more on the test set (0.93 → 0.92) because the test set has only 77 rows for 16 inputs, so the penalty for the number of inputs is larger. The gap is small, so the model still generalizes well.

## Question 4 (1 point)

Many of the 16 inputs measure almost the same thing. Two inputs that move together carry the same information, and the model cannot separate their effects. Answer parts (a) to (c), each in its own cell.

### Part (a) - Find the correlated inputs

Compute the correlation between the inputs of the **train** set. Then list every pair with an absolute correlation above 0.9. This code does it for you:

```Python
import numpy as np
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))   # keep each pair once
pairs = upper.stack()                                               # one row per pair
print(pairs[pairs.abs() > 0.9])
```

In the answer cell: name the pairs. Explain why each pair makes sense for walking. For example, why do `cadence` and `avg_step_time` move together?

In [ ]:
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))   # keep each pair once
pairs = upper.stack()                                               # one row per pair
print(pairs[pairs.abs() > 0.9])

**Your answer:** Five pairs have an absolute correlation above 0.9:

| Pair | r | Why it makes sense |
| --- | --- | --- |
| `avg_stride_time_left` – `avg_stance_time_left` | 0.94 | A stride is stance + swing. Stance is the larger part of the stride (about 60%), so when the stride gets longer, the stance gets longer too. |
| `avg_stride_time_right` – `avg_stance_time_right` | 0.93 | Same reason, for the right foot. |
| `avg_step_time` – `cadence` | −0.94 | Cadence is steps per minute, and step time is seconds per step. They are the same information turned upside down (cadence ≈ 60 / step time). Longer steps in time mean fewer steps per minute, so the correlation is negative. |
| `cv_stride_time_left` – `cv_stance_time_left` | 0.99 | The stride contains the stance, so when the stance time varies from step to step, the stride time varies too. The variability of one carries over to the other. |
| `cv_stride_time_right` – `cv_stance_time_right` | 0.99 | Same reason, for the right foot. |

Each pair measures nearly the same thing, so the model cannot tell their effects apart.

### Part (b) - Compare feature sets with cross-validation

Compare three feature sets. Use 5-fold cross-validation on the **train set** and the $R^2$ score. Report the mean and the standard deviation for each set.

```Python
drop_list = ['avg_stance_time_left', 'avg_stance_time_right', 'avg_step_time',
             'cv_stance_time_left', 'cv_stance_time_right']

feature_sets = {
    'all inputs'      : features,
    'no duplicates'   : [f for f in features if f not in drop_list],
    'step length + cadence' : ['avg_step_length', 'cadence'],
}
```

**Hint:** use `cross_val_score` from `sklearn.model_selection`, with `cv=5` and `scoring="r2"`, inside a `for` loop over `feature_sets`.

Choose the best feature set with this rule: take the set with the highest mean score. If another set scores within one standard deviation of it, pick the set with fewer inputs.

In the answer cell: which set did you choose, and why?

In [ ]:
from sklearn.model_selection import cross_val_score

drop_list = ['avg_stance_time_left', 'avg_stance_time_right', 'avg_step_time',
             'cv_stance_time_left', 'cv_stance_time_right']

feature_sets = {
    'all inputs'      : features,
    'no duplicates'   : [f for f in features if f not in drop_list],
    'step length + cadence' : ['avg_step_length', 'cadence'],
}

cv_results = []
for name, cols in feature_sets.items():
    scores = cross_val_score(LinearRegression(), X_train[cols], y_train, cv=5, scoring="r2")
    cv_results.append({'feature set': name, 'n inputs': len(cols),
                       'mean R2': scores.mean(), 'std R2': scores.std()})
    print(f'{name:<22} ({len(cols):>2} inputs): fold scores = {np.round(scores, 3)}')

cv_results = pd.DataFrame(cv_results).set_index('feature set')
display(cv_results.round(3))

# selection rule: best mean, but prefer fewer inputs if within one SD of the best
best = cv_results['mean R2'].idxmax()
threshold = cv_results.loc[best, 'mean R2'] - cv_results.loc[best, 'std R2']
candidates = cv_results[cv_results['mean R2'] >= threshold]
chosen_set = candidates['n inputs'].idxmin()
print(f'highest mean: {best}; threshold = {threshold:.3f}')
print(f'chosen feature set: {chosen_set}')

**Your answer:** I chose **step length + cadence**.

- The highest mean cross-validated $R^2$ is from **no duplicates**: 0.891 ± 0.168 (11 inputs).
- **All inputs** scores 0.882 ± 0.187 (16 inputs), and **step length + cadence** scores 0.866 ± 0.159 (2 inputs).
- Both are within one standard deviation of the best set: anything above 0.891 − 0.168 ≈ 0.724 counts.
- By the rule, among the sets that are statistically about as good, I pick the one with the fewest inputs, which is step length + cadence with only 2 inputs.

The difference in mean score (0.89 vs 0.87) is tiny compared with the variation between folds. One fold scores only about 0.5–0.55 for every set, probably because it contains the unusual recordings. So the extra 9–14 inputs do not give a reliable improvement.

### Part (c) - Evaluate the final model

Train a model on the **whole train set** with the feature set you chose. Compute the $R^2$ score and the adjusted $R^2$ score on the test set.

In the answer cell: compare this model with the models from Questions 2 and 3. Is it the best model on the test set? Why is it still a good choice?

In [ ]:
final_features = feature_sets[chosen_set]

model_final = LinearRegression()
model_final.fit(X_train[final_features], y_train)

r2_test_final  = model_final.score(X_test[final_features], y_test)
adj_test_final = calculate_adjusted_r2(r2_test_final, X_test.shape[0], len(final_features))

print('features:', final_features)
print(f'test: R2 = {r2_test_final:.3f}, adjusted R2 = {adj_test_final:.3f}')

summary = pd.DataFrame({
    'n inputs':       [1, len(features), len(final_features)],
    'test R2':        [r2_test_q2, r2_test_q3, r2_test_final],
    'test adjusted R2': [calculate_adjusted_r2(r2_test_q2, X_test.shape[0], 1), adj_test_q3, adj_test_final],
}, index=['Q2: step length only', 'Q3: all inputs', 'Q4: step length + cadence'])
summary.round(3)

**Your answer:** On the test set, the final model (step length + cadence) has $R^2$ = **0.941** and adjusted $R^2$ = **0.939**. The other models scored:

- Q2, step length only: $R^2$ = 0.710, adjusted 0.706.
- Q3, all 16 inputs: $R^2$ = 0.934, adjusted 0.917.

So with this split, the 2-input model is actually slightly the best on the test set, even on plain $R^2$, and it is clearly the best on adjusted $R^2$. But the difference from the all-inputs model is small (0.941 vs 0.934). It could easily flip with a different random split, so the test score alone is not the reason to choose it.

It is still a good choice because:

- **It is much simpler:** 2 inputs instead of 16, with the same accuracy. It is easier to interpret, needs fewer measurements, and has less room to overfit. The adjusted $R^2$ rewards this.
- **It avoids duplicated inputs:** the strongly correlated pairs from part (a) are not in the model.
- **It matches the physics of walking:** speed = cadence × step length / 60, so these two inputs carry almost all the information needed to predict speed (see Assignment 4).

## Question 5 (1 point)

A model can also fail because it is too flexible. In lecture 03.02, a high-order polynomial fitted the training data almost perfectly and failed on new data. Here you repeat that experiment with real gait data. Answer parts (a) and (b), each in its own cell.

Use one input, `avg_step_length`, and a small train set of 30 rows. Small sets make overfitting easy to see. Run the cell below to create the small set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.RandomState(15)
idx = rng.choice(len(X_train), 30, replace=False)          # 30 random rows of the train set
x_small = X_train[['avg_step_length']].iloc[idx]
y_small = np.asarray(y_train)[idx]
print(x_small.shape, y_small.shape)

### Part (a) - Train and validation score for each polynomial order

For every order from 1 to 10:

1. Build this pipeline. It creates the polynomial inputs and then fits a linear model.
2. Run 5-fold cross-validation on `x_small` and `y_small`. Keep the train scores and the validation scores.
3. Store the mean train score and the mean validation score.

```Python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_validate

pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=order, include_bias=True)),
    ("lr", LinearRegression())
])
results = cross_validate(pipe, x_small, y_small, cv=5, scoring="r2", return_train_score=True)
# results['train_score'] and results['test_score'] hold one value per fold
```

Then plot both scores against the order, with a legend. Add `plt.ylim(-1, 1)` to the plot. Without it, the validation scores of the high orders are so negative that you cannot see the rest.

In the answer cell: which order gives the best validation score? Describe what happens to the train score and to the validation score as the order grows. Where does the model underfit, and where does it overfit?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_validate

orders = range(1, 11)
mean_train_scores = []
mean_val_scores = []

for order in orders:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=order, include_bias=True)),
        ("lr", LinearRegression())
    ])
    results = cross_validate(pipe, x_small, y_small, cv=5, scoring="r2", return_train_score=True)
    mean_train_scores.append(results['train_score'].mean())
    mean_val_scores.append(results['test_score'].mean())

poly_scores = pd.DataFrame({'mean train R2': mean_train_scores, 'mean validation R2': mean_val_scores},
                           index=pd.Index(orders, name='order'))
display(poly_scores.round(3))

best_order = poly_scores['mean validation R2'].idxmax()
print('order with the best validation score:', best_order)

plt.figure(figsize=(7, 5))
plt.plot(orders, mean_train_scores, 'o-', label='Train score')
plt.plot(orders, mean_val_scores, 's--', label='Validation score')
plt.ylim(-1, 1)
plt.xticks(list(orders))
plt.xlabel('Polynomial order')
plt.ylabel('Mean $R^2$ (5-fold cross-validation)')
plt.title('Speed from step length (30 training rows)')
plt.legend()
plt.grid()
plt.show()

**Your answer:** **Order 3** gives the best mean validation score ($R^2$ = 0.69). Order 2 is very close (0.68).

- **Train score:** rises slowly as the order grows, from 0.81 at order 1 to about 0.85 at orders 7–10. A more flexible curve can always bend closer to the training points, so the train score never goes down.
- **Validation score:** rises a little from order 1 (0.65) to order 3 (0.69), then drops (0.51 at order 4). From order 5 on, it collapses to large negative values: −13 at order 5 and −630 at order 10. These high-order models are far worse than just predicting the mean speed.
- **Underfitting:** order 1 underfits slightly. A straight line is a bit too simple, and both its train and validation scores are lower than at orders 2–3. The difference is small, because speed vs step length is almost linear.
- **Overfitting:** from **order 4**, and badly from order 5. The train score keeps improving while the validation score falls apart. With only 30 rows, the high-order polynomial follows the noise in the training points and swings wildly between and beyond them, especially at the edges of the data.

### Part (b) - Check on the test set

Train two models on `x_small` and `y_small`: one with the order you chose in part (a), and one with order 10. Compute the $R^2$ score of each model on the train rows (`x_small`) and on the **whole test set** (`X_test[['avg_step_length']]`).

In the answer cell: compare the four scores. What would you have believed about the order-10 model if you had only looked at its train score?

In [ ]:
X_test_step = X_test[['avg_step_length']]
q5_results = {}

for order in [best_order, 10]:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=order, include_bias=True)),
        ("lr", LinearRegression())
    ])
    pipe.fit(x_small, y_small)
    q5_results[f'order {order}'] = {'train R2 (x_small)': pipe.score(x_small, y_small),
                                    'test R2 (whole test set)': pipe.score(X_test_step, y_test)}

pd.DataFrame(q5_results).T.round(3)

**Your answer:**

| Model | Train $R^2$ (30 rows) | Test $R^2$ (77 rows) |
| --- | --- | --- |
| Order 3 | 0.832 | 0.690 |
| Order 10 | 0.840 | about −21,000 |

On the train rows the two models look almost identical, and order 10 even scores slightly *higher* (0.840 vs 0.832). If I had only looked at the train score, I would have believed the order-10 model was the best model.

On the test set, the order-3 model keeps a reasonable score (0.69), while the order-10 model fails completely, with a hugely negative $R^2$. Its predictions are useless, far worse than just guessing the mean speed. Part of the reason is that some test recordings have step lengths outside the range of the 30 training rows (0.38–0.90 m), for example 0.32 m and 1.16 m. There, the order-10 curve shoots off to extreme values. The train score cannot show this kind of overfitting; only data the model has not seen can.

## Question 6 (1 points)

Which input has more influence on the speed: step length or cadence? Train a `LinearRegression` on the train set with these two inputs only. Answer parts (a) and (b), each in its own cell.

### Part (a) - Coefficients in original units

Print the two coefficients and the $R^2$ score on the test set.

In the answer cell: can you say which input matters more from these coefficients? Why not? (Look at the units of each input.)

In [ ]:
two_inputs = ['avg_step_length', 'cadence']

model_q6 = LinearRegression()
model_q6.fit(X_train[two_inputs], y_train)

for name, coef in zip(two_inputs, model_q6.coef_):
    print(f'coefficient of {name}: {coef:.4f}')
print(f'intercept: {model_q6.intercept_:.4f}')
print(f'R2 on the test set: {model_q6.score(X_test[two_inputs], y_test):.3f}')

**Your answer:** The coefficients are **1.511 for step length** and **0.0099 for cadence** (test $R^2$ = 0.941). The step-length coefficient looks about 150 times larger, but **we cannot conclude from this that step length matters more**, because the two coefficients are in different units:

- Step length: 1.511 (m/s) per **metre**. Step length varies over only a few tenths of a metre (SD ≈ 0.13 m).
- Cadence: 0.0099 (m/s) per **step/min**. Cadence varies over tens of steps per minute (SD ≈ 14 steps/min).

A change of 1 m in step length is enormous, while a change of 1 step/min in cadence is tiny, so the raw coefficients are not comparable. If cadence were measured in steps per second, its coefficient would be 60 times bigger, even though nothing about walking had changed.

### Part (b) - Coefficients after scaling

Scaling removes the mean of each input and divides by its standard deviation, so all inputs have the same scale. Repeat the fit with this pipeline:

```Python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
# after pipe.fit(...), the coefficients are in pipe.named_steps["lr"].coef_
```

Print the two coefficients and the $R^2$ score on the test set.

In the answer cell: which input has more influence now? Did the $R^2$ score change? Did the predictions change? Explain in one or two sentences why scaling helps here.

In [ ]:
from sklearn.preprocessing import StandardScaler

pipe_q6 = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
pipe_q6.fit(X_train[two_inputs], y_train)

for name, coef in zip(two_inputs, pipe_q6.named_steps["lr"].coef_):
    print(f'scaled coefficient of {name}: {coef:.4f} (m/s per 1 SD of the input)')
print(f'R2 on the test set: {pipe_q6.score(X_test[two_inputs], y_test):.3f}')

# check whether the predictions changed compared with the unscaled model
diff = np.abs(pipe_q6.predict(X_test[two_inputs]) - model_q6.predict(X_test[two_inputs])).max()
print(f'largest difference between scaled and unscaled predictions: {diff:.2e} m/s')

**Your answer:** After scaling, the coefficients are **0.203 for step length** and **0.142 for cadence**. These are in m/s per one standard deviation of each input, so they can be compared directly. **Step length has more influence**: a one-SD change in step length changes the predicted speed about 1.4 times as much as a one-SD change in cadence.

The $R^2$ on the test set **did not change** (0.941), and the **predictions did not change** either. They are identical to the unscaled model (the largest difference is about 10⁻¹⁶ m/s, which is just rounding). Scaling only changes the units the inputs are expressed in, so the fitted line is the same.

Scaling helps because it puts both inputs on the same scale (mean 0, SD 1). Each coefficient then means "change in speed for a typical-sized change in this input", so comparing coefficients compares influence, not units.